# Traffic Integrity: Web Bots and Network Coordination Graphs

This version downloads and parses the official Bournemouth/M4D **phase 1** archive by default. It trains on the actual Apache access logs joined to the published PHP-session annotations: **human (0), moderate bot (1), advanced bot (2)**. It does not assign malicious intent, legitimate-crawler status, or benign-automation status to these labels.

The dataset contains 50 sessions per class. The loader joins roughly 29,000 labeled access-log rows, then retains about 2,000 navigation requests after excluding mouse-movement telemetry and static assets. The notebook uses a stratified **session-disjoint** train/validation/test split. The classes were collected on different dates, so a chronological holdout on phase 1 cannot support a meaningful three-class comparison.

Source: https://m4d.iti.gr/web-bot-detection-dataset/ (CC BY-NC-SA). The ZIP is fetched on first run. An existing local ZIP can be supplied in `CFG.data_path`.

The later sections run a synthetic four-class chronological integration test and offer a strict gate for separately collected reviewed real data. The network coordination fixture is synthetic.


## Changes from the ETA notebook and the synthetic Traffic Integrity notebook

The taxi ingestion is replaced by the actual phase 1 Apache logs and session annotations. Session windows feed the tabular and request-sequence models. The graph connects windows from the **same session**, with path overlap and timing as edge attributes. This is a behavioral context graph; the published logs do not provide trustworthy IP, ASN, TLS, or campaign identity for Sybil coordination.

Training, calibration, policy thresholds, and the synthetic evasion probe operate on disjoint sessions. The latter probe modifies held-out bot requests and is an experimental stress test, not a new observed attack dataset.


## 1. Environment

Run the next cell once in Colab. If your runtime already has the packages, it exits quickly. PyTorch must be installed before PyTorch Geometric.


In [ ]:
# Install missing packages in a fresh Colab/local runtime.
import importlib
import importlib.util
import site
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "xgboost": "xgboost", "torch": "torch", "torch_geometric": "torch-geometric",
}
missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()
else:
    print("All required packages are already installed.")


In [ ]:
import copy
import math
import random
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, classification_report,
    confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import MessagePassing, SAGEConv
    from torch_geometric.utils import softmax as pyg_softmax
    PYG_AVAILABLE = True
except ImportError:
    PYG_AVAILABLE = False
    print("PyTorch Geometric is unavailable; tabular and sequence sections still run.")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| PyG:", PYG_AVAILABLE)


In [ ]:
@dataclass
class Config:
    data_path: str = "web_bot_detection_dataset.zip"
    dataset_url: str = "https://m4d.iti.gr/wp-content/uploads/2024/04/web_bot_detection_dataset.zip"
    window_minutes: int = 5
    sequence_length: int = 48
    batch_size: int = 128
    sequence_epochs: int = 4
    graph_epochs: int = 30
    max_group_members: int = 20
    target_fpr: float = 0.01
    quick_run: bool = True

CFG = Config()
LABEL_NAMES = {0: "human", 1: "moderate_bot", 2: "advanced_bot"}
ABUSIVE_CLASS = 2  # Legacy variable name: the positive class is advanced bot, not proven abuse.


## 2. Official archive and canonical request schema

The original phase 1 annotations provide 50 labeled PHP sessions for each class. The loader ignores entries without a valid session ID or published annotation and reports parsed/dropped counts. It then retains navigation paths, excluding mouse telemetry endpoints, JavaScript/CSS and favicon. Network and latency fields absent from Apache logs are explicitly marked unavailable and are excluded from model features and graph relations.


In [ ]:
CANONICAL_COLUMNS = [
    "timestamp", "request_id", "session_id", "client_id", "ip_hash",
    "subnet", "asn", "country", "user_agent", "tls_fingerprint",
    "path", "method", "status_code", "response_bytes", "latency_ms",
    "payload_bytes", "label",
]


def shannon_entropy(values):
    p = pd.Series(values).value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p + 1e-12)).sum())


def validate_request_schema(df):
    missing = sorted(set(CANONICAL_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing canonical columns: {missing}")
    out = df[CANONICAL_COLUMNS].copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")
    out["label"] = pd.to_numeric(out["label"], errors="raise").astype(int)
    if out["timestamp"].isna().any():
        raise ValueError("Some timestamps could not be parsed")
    if not set(out["label"].unique()).issubset(LABEL_NAMES):
        raise ValueError("Labels must be integers 0..2")
    return out.sort_values("timestamp").reset_index(drop=True)


In [ ]:
def generate_controlled_traffic(n_requests=30_000, seed=42):
    '''Generate illustrative synthetic traffic for an end-to-end smoke test.

    This is not presented as production evidence. It is a controlled benchmark that
    makes class semantics, temporal drift, coordination, and evasions explicit.
    '''
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2026-01-01", tz="UTC")
    class_probs = np.array([0.64, 0.10, 0.16, 0.10])
    labels = rng.choice(4, size=n_requests, p=class_probs)

    profiles = {
        0: dict(rate=0.10, jitter=1.20, err=0.025, latency=180, payload=850, paths=18),
        1: dict(rate=0.45, jitter=0.45, err=0.015, latency=95, payload=160, paths=30),
        2: dict(rate=0.28, jitter=0.30, err=0.040, latency=120, payload=1100, paths=10),
        3: dict(rate=1.60, jitter=0.10, err=0.180, latency=70, payload=320, paths=42),
    }
    path_pool = ["/", "/search", "/login", "/api/items", "/api/profile", "/cart"]
    path_pool += [f"/product/{i}" for i in range(80)]
    human_uas = ["Chrome-Win", "Safari-iOS", "Chrome-Android", "Firefox-Linux"]
    crawler_uas = ["Googlebot", "Bingbot", "UptimeMonitor"]
    benign_uas = ["PartnerAPI/2.1", "InternalJob/4.0", "MobileSDK/8.2"]
    abusive_uas = ["python-requests", "HeadlessChrome", "curl/8", "Chrome-Win"]
    ua_pool = {0: human_uas, 1: crawler_uas, 2: benign_uas, 3: abusive_uas}

    # Clients deliberately share infrastructure inside automation campaigns.
    clients_by_class = {k: [f"c{k}_{i:04d}" for i in range(v)]
                        for k, v in {0: 620, 1: 45, 2: 90, 3: 75}.items()}
    client_clock = {
        client: start + pd.Timedelta(days=float(rng.uniform(0, 175)))
        for members in clients_by_class.values() for client in members
    }
    rows = []
    for i, y in enumerate(labels):
        p = profiles[int(y)]
        client = rng.choice(clients_by_class[int(y)])
        gap = rng.lognormal(mean=np.log(1 / p["rate"] + 1e-3), sigma=p["jitter"])
        client_clock[client] += pd.Timedelta(seconds=float(gap))
        ts = client_clock[client]
        campaign = int(client.split("_")[1]) // 10
        subnet = f"10.{int(y)}.{campaign}.0/24"
        ip = f"{subnet[:-4]}{rng.integers(1, 250)}"
        ua = rng.choice(ua_pool[int(y)])
        tls = f"tls-{int(y)}-{campaign if y in (1, 3) else rng.integers(0, 100)}"
        available = path_pool[:p["paths"]]
        path = rng.choice(available)
        status = int(rng.choice([200, 302, 400, 401, 403, 404, 429, 500],
                                p=[1-p["err"]-.07, .05, .01, .01, p["err"]*.25,
                                   p["err"]*.35, p["err"]*.35, p["err"]*.05]))
        session_bucket = int(ts.timestamp() // (30 * 60))
        rows.append({
            "timestamp": ts, "request_id": f"r{i:08d}",
            "session_id": f"{client}-s{session_bucket}", "client_id": client,
            "ip_hash": ip, "subnet": subnet, "asn": f"AS{1000 + int(y)*20 + campaign}",
            "country": rng.choice(["US", "CA", "GB", "DE", "IN"], p=[.62,.08,.10,.08,.12]),
            "user_agent": ua, "tls_fingerprint": tls, "path": path,
            "method": rng.choice(["GET", "POST", "PUT"], p=[.86,.12,.02]),
            "status_code": status,
            "response_bytes": max(50, int(rng.lognormal(np.log(p["payload"]*3), .55))),
            "latency_ms": max(2, float(rng.lognormal(np.log(p["latency"]), .40))),
            "payload_bytes": max(0, int(rng.lognormal(np.log(p["payload"]), .65))),
            "label": int(y),
        })
    return validate_request_schema(pd.DataFrame(rows))


In [ ]:
import re
import zipfile
import urllib.request
from collections import Counter

LOG_RE = re.compile(
    r'^\S+ \S+ \[(?P<timestamp>[^]]+)\] "(?P<method>\S+) '
    r'(?P<path>.*?) HTTP/[^"]+" (?P<status>\d{3}) (?P<size>\S+) '
    r'"[^"]*" (?P<session>\S+) "(?P<ua>.*)"$'
)

archive_path = Path(CFG.data_path)
if not archive_path.exists():
    print("Downloading official Bournemouth dataset to", archive_path)
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CFG.dataset_url, archive_path)

def load_bournemouth_phase1(path):
    with zipfile.ZipFile(path) as z:
        annotations = {}
        for comparison in ("humans_and_moderate_bots", "humans_and_advanced_bots"):
            for partition in ("train", "test"):
                key = f"phase1/annotations/{comparison}/{partition}"
                for line in z.read(key).decode("utf-8").splitlines():
                    fields = line.split()
                    if len(fields) != 2:
                        continue
                    sid, kind = fields
                    if sid in annotations and annotations[sid] != kind:
                        raise ValueError(f"Conflicting annotation for session {sid}")
                    annotations[sid] = kind
        label_map = {"human": 0, "moderate_bot": 1, "advanced_bot": 2}
        rows = []
        counts = Counter()
        for name in z.namelist():
            if not (name.startswith("phase1/data/web_logs/") and name.endswith(".log")):
                continue
            for line in z.read(name).decode("utf-8", errors="replace").splitlines():
                match = LOG_RE.match(line)
                if match is None:
                    counts["unparsed"] += 1
                    continue
                d = match.groupdict()
                sid = d["session"]
                if sid not in annotations:
                    counts["unlabeled_or_missing_session"] += 1
                    continue
                kind = annotations[sid]
                timestamp = pd.to_datetime(d["timestamp"], format="%d/%b/%Y:%H:%M:%S %z", utc=True)
                rows.append({
                    "timestamp": timestamp, "request_id": f"r{len(rows):08d}",
                    "session_id": sid, "client_id": sid, "ip_hash": "unknown-ip",
                    "subnet": "unknown-subnet", "asn": "unknown-asn", "country": "unknown",
                    "user_agent": d["ua"], "tls_fingerprint": "unknown-tls",
                    "path": d["path"].split("?", 1)[0], "method": d["method"],
                    "status_code": int(d["status"]),
                    "response_bytes": int(d["size"]) if d["size"].isdigit() else 0,
                    "latency_ms": 0.0, "payload_bytes": 0, "label": label_map[kind],
                })
                counts["labeled"] += 1
    out = validate_request_schema(pd.DataFrame(rows))
    counts["labeled_all_http"] = len(out)
    navigation = out.path.eq("/") | out.path.eq("/index.php") | out.path.str.startswith("/content/")
    out = out.loc[navigation].reset_index(drop=True)
    counts["navigation"] = len(out)
    counts["telemetry_or_assets_excluded"] = counts["labeled_all_http"] - len(out)
    observed = out.groupby("session_id").label.first().value_counts().sort_index()
    print("Parsing counts:", dict(counts))
    print("Labeled sessions by class:", observed.to_dict())
    if observed.to_dict() != {0: 50, 1: 50, 2: 50}:
        raise ValueError("Expected 50 annotated sessions per class after navigation filtering")
    if out.session_id.isna().any() or out.request_id.duplicated().any():
        raise ValueError("Invalid or duplicate request identifiers")
    if (out.groupby("session_id").label.nunique() != 1).any():
        raise ValueError("A session received conflicting labels")
    return out

requests = load_bournemouth_phase1(archive_path)
display(requests.head())
display(requests["label"].map(LABEL_NAMES).value_counts())


### Synthetic campaign flow ETL

This reproducible fixture includes benign shared networks, popular destinations, rotating attack addresses, time jitter and daily campaign truth. Campaign IDs are reserved for evaluation and excluded from graph inputs. Documentation IP ranges are used. Synthetic scores do not measure real Sybil performance.


In [ ]:
SYNTHETIC_CAMPAIGN_PATH = Path("data/synthetic_campaign_flows.csv")

def generate_campaign_flows(path=SYNTHETIC_CAMPAIGN_PATH, seed=20260928):
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2026-01-01T08:00:00Z")
    popular = [f"192.0.2.{i}" for i in range(1, 17)]
    targets = [f"198.51.100.{i}" for i in range(1, 11)]
    for day in range(10):
        for window_id in range(30):
            window = start + pd.Timedelta(days=day, minutes=5*window_id)
            for client in range(int(rng.integers(5, 9))):
                src = f"100.64.{client % 12}.{int(rng.integers(1, 125))}"
                for _ in range(int(rng.integers(1, 4))):
                    dest = rng.choice(popular if rng.random() < .92 else targets)
                    rows.append((src, dest, window + pd.Timedelta(seconds=int(rng.integers(0,300))),
                                 int(rng.choice([80,443,8080], p=[.15,.82,.03])),
                                 int(rng.lognormal(2.5,.8)), 0, "none"))
            if window_id % 3 != 2 and rng.random() < .83:
                for identity in range(int(rng.integers(4,8))):
                    # Address pools avoid collisions while benign and attack sources share /24s.
                    src = f"100.64.{int(rng.integers(0,12))}.{int(rng.integers(130,250))}"
                    for _ in range(int(rng.integers(2,5))):
                        dest = targets[day] if rng.random() < .75 else rng.choice(popular)
                        rows.append((src, dest, window + pd.Timedelta(seconds=int(rng.integers(0,300))),
                                     int(rng.choice([80,443,8080],p=[.12,.82,.06])),
                                     int(rng.lognormal(2.8,.8)), 1, f"campaign_day_{day:02d}"))
    frame = pd.DataFrame(rows, columns=["src_ip","dst_ip","timestamp","dst_port","packets","label","campaign_id"])
    frame = frame.sort_values("timestamp").reset_index(drop=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path,index=False)
    assert set(frame.label)=={0,1} and frame.loc[frame.label.eq(1),"campaign_id"].nunique()==10
    return frame

synthetic_flows = generate_campaign_flows()
print("Synthetic flows:",len(synthetic_flows),"attack campaigns:",
      synthetic_flows.loc[synthetic_flows.label.eq(1),"campaign_id"].nunique())


## 3. Session-disjoint split

The annotation is per PHP session. All windows from a session remain in one split. We stratify sessions to preserve all three classes. This is a within-dataset benchmark; collection date is confounded with class and a chronological split would give an invalid comparison.


In [ ]:
from sklearn.model_selection import train_test_split

def add_window_id(df, minutes=5):
    out = df.copy()
    out["window_start"] = out["timestamp"].dt.floor(f"{minutes}min")
    out["node_id"] = out["session_id"].astype(str) + "|" + out["window_start"].astype(str)
    return out

requests = add_window_id(requests, CFG.window_minutes)
sessions = requests.groupby("session_id", as_index=False).label.first()
train_ids, remainder_ids = train_test_split(
    sessions.session_id.to_numpy(), test_size=0.4, random_state=SEED,
    stratify=sessions.label.to_numpy()
)
remainder = sessions.set_index("session_id").loc[remainder_ids]
val_ids, test_ids = train_test_split(
    remainder_ids, test_size=0.5, random_state=SEED,
    stratify=remainder.label.to_numpy()
)
request_splits = {
    split: requests[requests.session_id.isin(ids)].copy()
    for split, ids in (("train", train_ids), ("val", val_ids), ("test", test_ids))
}
assert len(set(train_ids) & set(val_ids)) == len(set(train_ids) & set(test_ids)) == len(set(val_ids) & set(test_ids)) == 0
for name, frame in request_splits.items():
    print(name, len(frame), "requests,", frame.session_id.nunique(), "sessions",
          frame.groupby("session_id").label.first().value_counts().sort_index().to_dict())


In [ ]:
def safe_mode(s):
    m = s.mode()
    return m.iloc[0] if len(m) else "unknown"


def aggregate_windows(req):
    rows = []
    for node_id, g in req.sort_values("timestamp").groupby("node_id", sort=False):
        iat = g["timestamp"].diff().dt.total_seconds().dropna().to_numpy()
        mean_iat = float(iat.mean()) if len(iat) else CFG.window_minutes * 60
        std_iat = float(iat.std()) if len(iat) > 1 else 0.0
        mu, sigma = mean_iat, std_iat
        burstiness = (sigma - mu) / (sigma + mu + 1e-6)
        secs = g["timestamp"].dt.floor("s")
        max_1s = int(secs.value_counts().max())
        duration = max(1.0, (g.timestamp.max() - g.timestamp.min()).total_seconds())
        labels = g["label"].value_counts()
        label = int(labels.index[0])
        rows.append({
            "node_id": node_id, "window_start": g.window_start.iloc[0],
            "window_end": g.timestamp.max(), "client_id": g.client_id.iloc[0],
            "label": label, "request_count": len(g),
            "requests_per_second": len(g) / duration,
            "requests_per_minute": len(g) / duration * 60,
            "unique_paths": g.path.nunique(), "unique_methods": g.method.nunique(),
            "unique_sessions": g.session_id.nunique(), "unique_ips": g.ip_hash.nunique(),
            "unique_uas": g.user_agent.nunique(), "unique_tls": g.tls_fingerprint.nunique(),
            "mean_iat_s": mean_iat, "std_iat_s": std_iat,
            "iat_cv": std_iat / (mean_iat + 1e-6), "burstiness": burstiness,
            "max_requests_1s": max_1s, "session_duration_s": duration,
            "error_rate": (g.status_code >= 400).mean(),
            "rate_4xx": g.status_code.between(400, 499).mean(),
            "rate_5xx": (g.status_code >= 500).mean(),
            "mean_latency_ms": g.latency_ms.mean(), "std_latency_ms": g.latency_ms.std(ddof=0),
            "mean_response_bytes": g.response_bytes.mean(),
            "std_response_bytes": g.response_bytes.std(ddof=0),
            "mean_payload_bytes": g.payload_bytes.mean(),
            "path_entropy": shannon_entropy(g.path),
            "method_entropy": shannon_entropy(g.method),
            "status_entropy": shannon_entropy(g.status_code),
            "ua_entropy": shannon_entropy(g.user_agent),
            "dominant_subnet": safe_mode(g.subnet), "dominant_asn": safe_mode(g.asn),
            "dominant_country": safe_mode(g.country), "dominant_ua": safe_mode(g.user_agent),
            "dominant_tls": safe_mode(g.tls_fingerprint),
            "path_set": frozenset(g.path.astype(str)),
        })
    return pd.DataFrame(rows).sort_values("window_start").reset_index(drop=True)


node_splits = {name: aggregate_windows(frame) for name, frame in request_splits.items()}
for name, frame in node_splits.items():
    print(name, "nodes:", len(frame), "class counts:", frame.label.value_counts().to_dict())
if set(node_splits["train"].label) != set(LABEL_NAMES):
    raise ValueError("Training requires all three classes")
for split in ["val", "test"]:
    if (node_splits[split].label == ABUSIVE_CLASS).sum() == 0 or (node_splits[split].label != ABUSIVE_CLASS).sum() == 0:
        raise ValueError(f"{split} requires both abusive and non-abusive nodes for operating metrics")


## 4. Tabular baselines

The imputer, scaler, and one-hot encoder fit on training sessions only. No IP, ASN, TLS, country, latency, payload, or status-code features are used. Status is constant (200) in the published labeled data.


In [ ]:
NUMERIC_FEATURES = [
    "request_count", "requests_per_second", "requests_per_minute", "unique_paths",
    "unique_sessions", "unique_uas",
    "mean_iat_s", "std_iat_s", "iat_cv", "burstiness", "max_requests_1s",
    "session_duration_s",  "mean_response_bytes", "std_response_bytes", 
    "path_entropy", "ua_entropy",
]
CATEGORICAL_FEATURES = ["dominant_ua"]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
], verbose_feature_names_out=False)

X = {}
y = {}
X["train"] = preprocessor.fit_transform(node_splits["train"][MODEL_FEATURES])
for split in ["val", "test"]:
    X[split] = preprocessor.transform(node_splits[split][MODEL_FEATURES])
for split in node_splits:
    y[split] = node_splits[split].label.to_numpy()

print("feature matrix:", {k: v.shape for k, v in X.items()})


In [ ]:
def expected_calibration_error(y_true, probs, n_bins=15):
    confidence = probs.max(axis=1)
    prediction = probs.argmax(axis=1)
    correct = (prediction == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


def fpr_at_tpr(y_binary, score, target_tpr=0.90):
    if np.unique(y_binary).size < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_binary, score)
    eligible = np.where(tpr >= target_tpr)[0]
    return float(fpr[eligible[0]]) if len(eligible) else 1.0


def tpr_at_fpr(y_binary, score, target_fpr=0.01):
    if np.unique(y_binary).size < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_binary, score)
    eligible = np.where(fpr <= target_fpr)[0]
    return float(tpr[eligible[-1]]) if len(eligible) else 0.0


def multiclass_metrics(y_true, probs, model_name="model", split="test"):
    pred = probs.argmax(axis=1)
    binary = (y_true == ABUSIVE_CLASS).astype(int)
    p_abuse = probs[:, ABUSIVE_CLASS]
    per_class_roc, per_class_ap = [], []
    for class_id in LABEL_NAMES:
        target = (y_true == class_id).astype(int)
        if np.unique(target).size == 2:
            per_class_roc.append(roc_auc_score(target, probs[:, class_id]))
        if target.sum() > 0:
            per_class_ap.append(average_precision_score(target, probs[:, class_id]))
    return {
        "model": model_name, "split": split,
        "roc_auc_macro_ovr": float(np.mean(per_class_roc)) if per_class_roc else np.nan,
        "pr_auc_macro": float(np.mean(per_class_ap)) if per_class_ap else np.nan,
        "advanced_bot_pr_auc": average_precision_score(binary, p_abuse) if binary.sum() else np.nan,
        "tpr_at_1pct_fpr": tpr_at_fpr(binary, p_abuse, 0.01),
        "fpr_at_90pct_tpr": fpr_at_tpr(binary, p_abuse, 0.90),
        "ece": expected_calibration_error(y_true, probs),
        "advanced_bot_brier": brier_score_loss(binary, p_abuse),
        "accuracy": float((pred == y_true).mean()),
    }


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.1,
        class_weight="balanced", random_state=SEED,
    ),
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=350 if not CFG.quick_run else 120, max_depth=7,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", eval_metric="mlogloss", random_state=SEED,
        n_jobs=-1,
    )
except ImportError:
    print("xgboost not installed; continuing with sklearn baselines")

tabular_probabilities = {}
rows = []
for name, model in models.items():
    model.fit(X["train"], y["train"])
    for split in ["val", "test"]:
        probs = model.predict_proba(X[split])
        tabular_probabilities[(name, split)] = probs
        rows.append(multiclass_metrics(y[split], probs, name, split))

baseline_results = pd.DataFrame(rows).sort_values(["split", "advanced_bot_pr_auc"], ascending=[True, False])
display(baseline_results.round(4))


In [ ]:
best_tabular_name = (
    baseline_results[baseline_results.split.eq("val")]
    .sort_values("advanced_bot_pr_auc", ascending=False).iloc[0].model
)
best_tabular = models[best_tabular_name]
test_probs = tabular_probabilities[(best_tabular_name, "test")]
print("Selected on validation only:", best_tabular_name)
print(classification_report(y["test"], test_probs.argmax(1), labels=list(LABEL_NAMES),
                            target_names=LABEL_NAMES.values(), zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y["test"], test_probs.argmax(1)), annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set(title="Session-disjoint test confusion matrix", xlabel="Predicted", ylabel="Actual")
binary = (y["test"] == ABUSIVE_CLASS).astype(int)
precision, recall, _ = precision_recall_curve(binary, test_probs[:, ABUSIVE_CLASS])
ax[1].plot(recall, precision)
ax[1].set(title="Advanced-bot precision–recall", xlabel="Recall", ylabel="Precision", xlim=(0,1), ylim=(0,1))
plt.tight_layout()


## 5. Request-sequence Transformer

Per-window navigation sequences use URL path, user agent, interarrival time, and response size. Vocabulary and scaling fit on training sessions only.


In [ ]:
TOKEN_COLUMNS = ["path", "user_agent"]
SEQ_NUMERIC = ["delta_s", "response_bytes"]


def build_vocab(series, min_count=2):
    counts = series.astype(str).value_counts()
    return {token: i + 2 for i, token in enumerate(counts[counts >= min_count].index)}


train_req = request_splits["train"].sort_values("timestamp").copy()
vocabs = {c: build_vocab(train_req[c]) for c in TOKEN_COLUMNS}
seq_scaler = StandardScaler()
train_req["delta_s"] = train_req.groupby("client_id").timestamp.diff().dt.total_seconds().fillna(0).clip(0, 300)
seq_scaler.fit(train_req[SEQ_NUMERIC])


def make_sequences(req, nodes, max_len=48):
    req = req.sort_values("timestamp").copy()
    req["delta_s"] = req.groupby("client_id").timestamp.diff().dt.total_seconds().fillna(0).clip(0, 300)
    groups = {k: g for k, g in req.groupby("node_id")}
    cat_out = np.zeros((len(nodes), max_len, len(TOKEN_COLUMNS)), dtype=np.int64)
    num_out = np.zeros((len(nodes), max_len, len(SEQ_NUMERIC)), dtype=np.float32)
    mask_out = np.ones((len(nodes), max_len), dtype=bool)  # True means padding
    for i, node_id in enumerate(nodes.node_id):
        g = groups[node_id].tail(max_len)
        length = len(g)
        start = max_len - length
        for j, c in enumerate(TOKEN_COLUMNS):
            cat_out[i, start:, j] = [vocabs[c].get(str(v), 1) for v in g[c]]
        num_out[i, start:, :] = seq_scaler.transform(g[SEQ_NUMERIC]).astype(np.float32)
        mask_out[i, start:] = False
    return cat_out, num_out, mask_out, nodes.label.to_numpy()


sequence_arrays = {
    s: make_sequences(request_splits[s], node_splits[s], CFG.sequence_length)
    for s in ["train", "val", "test"]
}


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, arrays):
        self.cat, self.num, self.mask, self.y = arrays
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (
            torch.tensor(self.cat[i], dtype=torch.long),
            torch.tensor(self.num[i], dtype=torch.float),
            torch.tensor(self.mask[i], dtype=torch.bool),
            torch.tensor(self.y[i], dtype=torch.long),
        )


class RequestSequenceTransformer(nn.Module):
    def __init__(self, vocabs, d_model=96, nhead=4, layers=2, num_classes=len(LABEL_NAMES)):
        super().__init__()
        emb_dim = 16
        self.emb = nn.ModuleList([nn.Embedding(len(v) + 2, emb_dim, padding_idx=0) for v in vocabs.values()])
        self.input_proj = nn.Linear(emb_dim * len(vocabs) + len(SEQ_NUMERIC), d_model)
        self.position = nn.Parameter(torch.randn(1, CFG.sequence_length, d_model) * 0.02)
        block = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=192,
                                           dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, num_layers=layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, num_classes))

    def forward(self, cats, nums, padding_mask):
        pieces = [emb(cats[:, :, i]) for i, emb in enumerate(self.emb)]
        x = self.input_proj(torch.cat(pieces + [nums], dim=-1)) + self.position[:, :cats.size(1)]
        x = self.encoder(x, src_key_padding_mask=padding_mask)
        valid = (~padding_mask).unsqueeze(-1)
        pooled = (x * valid).sum(1) / valid.sum(1).clamp_min(1)
        return self.head(pooled)


def class_weights(labels):
    counts = np.bincount(labels, minlength=len(LABEL_NAMES))
    w = len(labels) / (len(LABEL_NAMES) * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float, device=DEVICE)


def predict_sequence(model, arrays):
    loader = DataLoader(SequenceDataset(arrays), batch_size=CFG.batch_size, shuffle=False)
    logits, labels = [], []
    model.eval()
    with torch.no_grad():
        for cats, nums, mask, yy in loader:
            logits.append(model(cats.to(DEVICE), nums.to(DEVICE), mask.to(DEVICE)).cpu())
            labels.append(yy)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()


In [ ]:
sequence_model = RequestSequenceTransformer(vocabs).to(DEVICE)
optimizer = torch.optim.AdamW(sequence_model.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights(y["train"]))
train_loader = DataLoader(SequenceDataset(sequence_arrays["train"]), batch_size=CFG.batch_size, shuffle=True)

best_state, best_val = None, float("inf")
for epoch in range(CFG.sequence_epochs):
    sequence_model.train()
    for cats, nums, mask, yy in train_loader:
        optimizer.zero_grad()
        logits = sequence_model(cats.to(DEVICE), nums.to(DEVICE), mask.to(DEVICE))
        loss = criterion(logits, yy.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sequence_model.parameters(), 1.0)
        optimizer.step()
    val_logits, val_labels = predict_sequence(sequence_model, sequence_arrays["val"])
    val_loss = F.cross_entropy(torch.tensor(val_logits), torch.tensor(val_labels)).item()
    if val_loss < best_val:
        best_val, best_state = val_loss, copy.deepcopy(sequence_model.state_dict())
    print(f"epoch={epoch+1} val_loss={val_loss:.4f}")

sequence_model.load_state_dict(best_state)
sequence_logits = {}
for split in ["val", "test"]:
    sequence_logits[split], _ = predict_sequence(sequence_model, sequence_arrays[split])
    display(pd.DataFrame([multiclass_metrics(y[split], softmax(sequence_logits[split], axis=1),
                                             "SequenceTransformer", split)]).round(4))


## 6. Session-context graph

Edges link adjacent five-minute windows of the **same PHP session**. Features describe path overlap, time separation, and request-rate similarity. This graph does not establish shared infrastructure or coordinated clients; those signals are unavailable in these logs.


In [ ]:
EDGE_NAMES = ["same_session", "path_jaccard", "temporal_similarity", "rate_similarity"]

def path_jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0

def build_semantic_graph(frame, feature_matrix):
    if not PYG_AVAILABLE:
        return None
    local = frame.reset_index(drop=True).copy()
    src, dst, attrs = [], [], []
    for _, group in local.groupby("client_id"):
        members = group.sort_values("window_start").index.to_list()
        for a, b in zip(members, members[1:]):
            ta, tb = local.loc[a], local.loc[b]
            dt = abs((ta.window_start - tb.window_start).total_seconds())
            rate_a, rate_b = ta.requests_per_minute, tb.requests_per_minute
            attr = [1.0, path_jaccard(ta.path_set, tb.path_set),
                    math.exp(-dt / 1800),
                    math.exp(-abs(rate_a - rate_b) / (abs(rate_a) + abs(rate_b) + 1e-3))]
            src.extend([a, b]); dst.extend([b, a]); attrs.extend([attr, attr])
    # Give isolated windows a self-loop; no links between separate sessions.
    incident = set(src) | set(dst)
    for i in range(len(local)):
        if i not in incident:
            src.append(i); dst.append(i); attrs.append([1.0, 1.0, 1.0, 1.0])
    return Data(
        x=torch.tensor(feature_matrix, dtype=torch.float),
        edge_index=torch.tensor([src, dst], dtype=torch.long),
        edge_attr=torch.tensor(attrs, dtype=torch.float),
        y=torch.tensor(local.label.to_numpy(), dtype=torch.long),
    )

graphs = {s: build_semantic_graph(node_splits[s], X[s]) for s in ["train", "val", "test"]}
if PYG_AVAILABLE:
    print({s: (g.num_nodes, g.num_edges) for s, g in graphs.items()})


In [ ]:
# ETL and graph invariants: these must pass before model training.
assert set(request_splits["train"].session_id).isdisjoint(request_splits["val"].session_id)
assert set(request_splits["train"].session_id).isdisjoint(request_splits["test"].session_id)
assert set(request_splits["val"].session_id).isdisjoint(request_splits["test"].session_id)
assert all(set(frame.label) == set(LABEL_NAMES) for frame in node_splits.values())
for split, frame in node_splits.items():
    assert len(frame) == len(X[split]) == len(y[split])
    assert np.array_equal(frame.label.to_numpy(), y[split])
    assert np.isfinite(X[split]).all()
    if PYG_AVAILABLE:
        graph = graphs[split]
        assert graph.num_nodes == len(frame)
        assert np.array_equal(graph.y.numpy(), y[split])
        source, target = graph.edge_index.numpy()
        assert (frame.client_id.to_numpy()[source] == frame.client_id.to_numpy()[target]).all()
print("ETL, split, feature, and graph invariants passed.")


In [ ]:
if PYG_AVAILABLE:
    class GraphSAGENet(nn.Module):
        def __init__(self, in_dim, hidden=96, num_classes=len(LABEL_NAMES)):
            super().__init__()
            self.conv1 = SAGEConv(in_dim, hidden)
            self.conv2 = SAGEConv(hidden, hidden)
            self.head = nn.Linear(hidden, num_classes)
        def forward(self, data):
            x = F.gelu(self.conv1(data.x, data.edge_index))
            x = F.dropout(x, 0.15, training=self.training)
            x = F.gelu(self.conv2(x, data.edge_index))
            return self.head(x)


    class EdgeAwareGraphTransformerLayer(MessagePassing):
        def __init__(self, hidden_dim=96, heads=4, edge_dim=len(EDGE_NAMES), dropout=0.1):
            super().__init__(aggr="add", node_dim=0)
            assert hidden_dim % heads == 0
            self.heads, self.dk = heads, hidden_dim // heads
            self.q = nn.Linear(hidden_dim, hidden_dim)
            self.k = nn.Linear(hidden_dim, hidden_dim)
            self.v = nn.Linear(hidden_dim, hidden_dim)
            self.edge_bias = nn.Linear(edge_dim, heads)
            self.out = nn.Linear(hidden_dim, hidden_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(hidden_dim)
            self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim*2), nn.GELU(),
                                     nn.Dropout(dropout), nn.Linear(hidden_dim*2, hidden_dim))
            self.dropout = dropout

        def forward(self, x, edge_index, edge_attr):
            residual = x
            q = self.q(x).view(-1, self.heads, self.dk)
            k = self.k(x).view(-1, self.heads, self.dk)
            v = self.v(x).view(-1, self.heads, self.dk)
            x = self.propagate(edge_index, q=q, k=k, v=v, edge_attr=edge_attr,
                               size=(x.size(0), x.size(0)))
            x = self.norm1(residual + F.dropout(self.out(x.reshape(-1, self.heads*self.dk)),
                                                self.dropout, self.training))
            return self.norm2(x + F.dropout(self.ffn(x), self.dropout, self.training))

        def message(self, q_i, k_j, v_j, edge_attr, index, ptr, size_i):
            score = (q_i * k_j).sum(-1) / math.sqrt(self.dk)
            score = score + self.edge_bias(edge_attr)
            alpha = pyg_softmax(score, index, ptr, size_i)
            return v_j * alpha.unsqueeze(-1)


    class EdgeAwareGraphTransformer(nn.Module):
        def __init__(self, in_dim, hidden=96, layers=2, heads=4, edge_dim=len(EDGE_NAMES), num_classes=len(LABEL_NAMES)):
            super().__init__()
            self.input = nn.Linear(in_dim, hidden)
            self.layers = nn.ModuleList([
                EdgeAwareGraphTransformerLayer(hidden, heads, edge_dim) for _ in range(layers)
            ])
            self.head = nn.Linear(hidden, num_classes)
        def forward(self, data):
            x = F.gelu(self.input(data.x))
            for layer in self.layers:
                x = layer(x, data.edge_index, data.edge_attr)
            return self.head(x)


In [ ]:
def train_full_graph(model, graphs, epochs=30):
    model = model.to(DEVICE)
    train_g, val_g = graphs["train"].to(DEVICE), graphs["val"].to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=5e-4)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(train_g.y.cpu().numpy()))
    best_state, best_val = None, float("inf")
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(train_g), train_g.y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(val_g), val_g.y).item()
        if val_loss < best_val:
            best_val, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


def graph_logits(model, graph):
    model.eval()
    with torch.no_grad():
        return model(graph.to(DEVICE)).cpu().numpy()


graph_models, graph_outputs = {}, {}
if PYG_AVAILABLE:
    in_dim = X["train"].shape[1]
    candidates = {
        "GraphSAGE": GraphSAGENet(in_dim),
        "EdgeGraphTransformer": EdgeAwareGraphTransformer(in_dim),
    }
    for name, candidate in candidates.items():
        graph_models[name] = train_full_graph(candidate, graphs, CFG.graph_epochs)
        for split in ["val", "test"]:
            logits = graph_logits(graph_models[name], graphs[split])
            graph_outputs[(name, split)] = logits
            display(pd.DataFrame([multiclass_metrics(y[split], softmax(logits, axis=1), name, split)]).round(4))
else:
    print("Install torch-geometric to run this section.")


## 7. Calibration and research routing

The positive score means **advanced bot in this dataset**. Thresholds are chosen on validation sessions. Routing labels are for analysis only: this dataset does not establish that advanced bots are abusive or justify real-world blocking.


In [ ]:
def fit_temperature(logits, labels):
    labels = np.asarray(labels, dtype=int)
    def nll(log_t):
        probs = softmax(logits / np.exp(log_t), axis=1)
        return -np.log(probs[np.arange(len(labels)), labels] + 1e-12).mean()
    result = minimize_scalar(nll, bounds=(-3, 3), method="bounded")
    return float(np.exp(result.x))


def probabilities_to_logits(probs):
    return np.log(np.clip(probs, 1e-8, 1.0))


all_model_logits = {}
for split in ["val", "test"]:
    all_model_logits[(best_tabular_name, split)] = probabilities_to_logits(
        tabular_probabilities[(best_tabular_name, split)]
    )
    all_model_logits[("SequenceTransformer", split)] = sequence_logits[split]
if PYG_AVAILABLE:
    all_model_logits.update(graph_outputs)

calibrated = {}
calibration_rows = []
model_names = sorted({name for name, split in all_model_logits if split == "val"})
for name in model_names:
    temp = fit_temperature(all_model_logits[(name, "val")], y["val"])
    for split in ["val", "test"]:
        probs = softmax(all_model_logits[(name, split)] / temp, axis=1)
        calibrated[(name, split)] = probs
        row = multiclass_metrics(y[split], probs, name + "+temperature", split)
        row["temperature"] = temp
        calibration_rows.append(row)

calibration_results = pd.DataFrame(calibration_rows)
display(calibration_results.round(4))


In [ ]:
selected_name = (
    calibration_results[calibration_results.split.eq("val")]
    .sort_values("advanced_bot_pr_auc", ascending=False).iloc[0].model.replace("+temperature", "")
)
val_abuse = calibrated[(selected_name, "val")][:, ABUSIVE_CLASS]
val_legit = y["val"] != ABUSIVE_CLASS
review_threshold = float(np.quantile(val_abuse[val_legit], 1 - CFG.target_fpr))
medium_threshold = max(0.50, review_threshold * 0.75)
monitor_threshold = max(0.30, review_threshold * 0.50)


def policy_action(p):
    if p >= review_threshold: return "REVIEW_HIGH"
    if p >= medium_threshold: return "REVIEW_MEDIUM"
    if p >= monitor_threshold: return "MONITOR"
    return "NO_FLAG"


test_abuse = calibrated[(selected_name, "test")][:, ABUSIVE_CLASS]
actions = pd.Series([policy_action(p) for p in test_abuse])
policy_report = pd.DataFrame({
    "metric": ["selected_model", "review_threshold", "non_advanced_high_review_rate",
               "advanced_high_review_rate", "advanced_flag_rate", "medium_or_monitor_share"],
    "value": [selected_name, review_threshold,
              (actions[y["test"] != ABUSIVE_CLASS] == "REVIEW_HIGH").mean(),
              (actions[y["test"] == ABUSIVE_CLASS] == "REVIEW_HIGH").mean(),
              (actions[y["test"] == ABUSIVE_CLASS] != "NO_FLAG").mean(),
              actions.isin(["REVIEW_MEDIUM", "MONITOR"]).mean()],
})
display(policy_report)
display(pd.crosstab(pd.Series(y["test"]).map(LABEL_NAMES), actions, normalize="index").round(3))


## 8. Controlled perturbation of held-out advanced-bot traffic

This probe removes some advanced-bot requests, adds timing jitter, rotates reported user agents, and diversifies paths. The perturbed requests are synthetic edits of real held-out logs. It measures sensitivity to those edits, not the performance against naturally evasive bots.


In [ ]:
def make_evasive_test_requests(test_req, seed=123):
    rng = np.random.default_rng(seed)
    out = test_req.copy()
    abusive = out.label.eq(ABUSIVE_CLASS)
    # Rate shaping: retain a random subset, but at least one request per node.
    keep = ~abusive | (rng.random(len(out)) > 0.45)
    first_per_node = out.groupby("node_id").head(1).index
    keep[out.index.isin(first_per_node)] = True
    out = out[keep].copy()
    abusive = out.label.eq(ABUSIVE_CLASS)
    jitter = rng.lognormal(mean=2.0, sigma=1.0, size=abusive.sum())
    out.loc[abusive, "timestamp"] += pd.to_timedelta(jitter, unit="s")
    out.loc[abusive, "user_agent"] = rng.choice(
        ["Chrome-Win", "Safari-iOS", "Chrome-Android", "Firefox-Linux"], abusive.sum()
    )
    diversify = rng.random(abusive.sum()) < 0.35
    idx = out.index[abusive][diversify]
    out.loc[idx, "path"] = rng.choice(["/", "/search", "/cart", "/api/profile"], len(idx))
    # Preserve original node membership so comparison is aligned to the same time window.
    return out.sort_values("timestamp")


evasive_requests = make_evasive_test_requests(request_splits["test"])
evasive_nodes = aggregate_windows(evasive_requests)
# Align on nodes that survive both datasets.
common = node_splits["test"].node_id[node_splits["test"].node_id.isin(evasive_nodes.node_id)]
clean_aligned = node_splits["test"].set_index("node_id").loc[common].reset_index()
evasive_aligned = evasive_nodes.set_index("node_id").loc[common].reset_index()
X_clean_aligned = preprocessor.transform(clean_aligned[MODEL_FEATURES])
X_evasive = preprocessor.transform(evasive_aligned[MODEL_FEATURES])
y_evasive = evasive_aligned.label.to_numpy()

clean_p = best_tabular.predict_proba(X_clean_aligned)
evasive_p = best_tabular.predict_proba(X_evasive)
robustness_rows = [
    multiclass_metrics(clean_aligned.label.to_numpy(), clean_p, best_tabular_name, "clean_aligned"),
    multiclass_metrics(y_evasive, evasive_p, best_tabular_name, "evasive"),
]

if PYG_AVAILABLE:
    clean_graph = build_semantic_graph(clean_aligned, X_clean_aligned)
    evasive_graph = build_semantic_graph(evasive_aligned, X_evasive)
    for graph_name, graph_model in graph_models.items():
        for split_name, graph, labels in [
            ("clean_aligned", clean_graph, clean_aligned.label.to_numpy()),
            ("evasive", evasive_graph, y_evasive),
        ]:
            probs = softmax(graph_logits(graph_model, graph), axis=1)
            robustness_rows.append(multiclass_metrics(labels, probs, graph_name, split_name))

robustness_results = pd.DataFrame(robustness_rows)
display(robustness_results[["model", "split", "advanced_bot_pr_auc", "tpr_at_1pct_fpr",
                            "fpr_at_90pct_tpr", "ece"]].round(4))


## 9. Inference latency

These are notebook-runtime measurements, not production service-level claims. Report hardware, batch size, warm-up, and sample count alongside p50/p95/p99.


In [ ]:
def latency_percentiles(callable_fn, warmup=10, repeats=100):
    for _ in range(warmup): callable_fn()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        callable_fn()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return dict(zip(["p50_ms", "p95_ms", "p99_ms"], np.percentile(times, [50, 95, 99])))


latency_rows = []
sample_x = X["test"][: min(256, len(X["test"]))]
latency_rows.append({"model": best_tabular_name, **latency_percentiles(lambda: best_tabular.predict_proba(sample_x))})

seq_batch = next(iter(DataLoader(SequenceDataset(sequence_arrays["test"]), batch_size=min(256, len(y["test"])))))
cats_b, nums_b, mask_b, _ = [z.to(DEVICE) for z in seq_batch]
sequence_model.eval()
latency_rows.append({"model": "SequenceTransformer", **latency_percentiles(
    lambda: sequence_model(cats_b, nums_b, mask_b), repeats=50
)})

if PYG_AVAILABLE:
    for name, model in graph_models.items():
        g = graphs["test"].to(DEVICE); model.eval()
        latency_rows.append({"model": name, **latency_percentiles(lambda m=model: m(g), repeats=50)})

latency_results = pd.DataFrame(latency_rows)
display(latency_results.round(3))


## 10. Final results and error analysis

These metrics are a within-site, session-disjoint benchmark. The number of independent sessions in test is small, so 1% FPR is below the resolution supported by this dataset. Avoid claiming production traffic-integrity performance from these results.


In [ ]:
final_rows = []
for name in model_names:
    row = multiclass_metrics(y["test"], calibrated[(name, "test")], name, "chronological_test")
    final_rows.append(row)
final_results = pd.DataFrame(final_rows).merge(latency_results, on="model", how="left")
display(final_results[[
    "model", "roc_auc_macro_ovr", "pr_auc_macro", "advanced_bot_pr_auc",
    "tpr_at_1pct_fpr", "fpr_at_90pct_tpr", "ece", "advanced_bot_brier",
    "p50_ms", "p95_ms", "p99_ms",
]].sort_values("advanced_bot_pr_auc", ascending=False).round(4))


In [ ]:
best_probs = calibrated[(selected_name, "test")]
errors = node_splits["test"].copy()
errors["predicted"] = best_probs.argmax(1)
errors["p_advanced_bot"] = best_probs[:, ABUSIVE_CLASS]
errors["correct"] = errors.label.eq(errors.predicted)
errors["true_name"] = errors.label.map(LABEL_NAMES)
errors["pred_name"] = errors.predicted.map(LABEL_NAMES)

display(errors.loc[~errors.correct, [
    "window_start", "client_id", "true_name", "pred_name", "p_advanced_bot",
    "request_count", "requests_per_minute", "path_entropy", "error_rate",
    "dominant_ua", "dominant_asn",
]].sort_values("p_advanced_bot", ascending=False).head(30))


## 11. Separate network-flow and IoT benchmarks

The four datasets below answer **network intrusion** questions. They are separate from Bournemouth's web-session classification. The optional runner uses a binary normal/attack task on each source, reports each result under its own dataset name, and never imports its labels into the web-bot model.

| Dataset | Official source | Expected local input |
|---|---|---|
| CSE-CIC-IDS2018 | https://www.unb.ca/cic/datasets/ids-2018.html | One or more CICFlowMeter CSV files with `Label` |
| BoT-IoT | https://research.unsw.edu.au/projects/bot-iot-dataset | CSV extract with `attack` (0/1) |
| UNSW-NB15 | https://research.unsw.edu.au/projects/unsw-nb15-dataset | Official `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv` |
| TON_IoT | https://research.unsw.edu.au/projects/toniot-datasets | **Network** CSV with `label` (0/1); do not mix telemetry/OS files |

Set paths below to CSVs you have obtained from those sources. The large archives are not downloaded automatically. No result is displayed as measured for a dataset until its files are present. The row sampler is for an exploratory baseline; a research result should process the full data and honor scenario, host, and time separation.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

SECONDARY_PATHS = {
    "CSE-CIC-IDS2018": [],  # e.g. ["data/cic/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv"]
    "BoT-IoT": [],           # e.g. ["data/bot_iot/5_percent.csv"]
    "UNSW-NB15": [],         # [official_training_set.csv, official_testing_set.csv]
    "TON_IoT": [],           # e.g. ["data/ton_iot/Train_Test_Network.csv"]
}

def _binary_labels(frame, name):
    columns = {str(c).strip().lower(): c for c in frame.columns}
    preferred = ["attack", "label"] if name == "BoT-IoT" else ["label", "attack"]
    key = next((columns[c] for c in preferred if c in columns), None)
    if key is None:
        raise ValueError(f"{name}: expected label/attack column, got {list(frame.columns)[:20]}")
    raw = frame[key].astype(str).str.strip().str.lower()
    normal = raw.isin(["0", "0.0", "benign", "normal", "normal."])
    attack = raw.isin(["1", "1.0"]) if name != "CSE-CIC-IDS2018" else ~normal
    if name != "CSE-CIC-IDS2018":
        unknown = ~(normal | attack)
        if unknown.any():
            raise ValueError(f"{name}: nonbinary {key} values; inspect label semantics")
    return np.where(normal, 0, 1).astype(int), key

def _read_csv_samples(paths, name, max_per_class=5000):
    chunks = []
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f"{name}: missing {path}")
        for chunk in pd.read_csv(path, chunksize=100_000, low_memory=False):
            chunk.columns = [str(c).strip() for c in chunk.columns]
            labels, _ = _binary_labels(chunk, name)
            chunk = chunk.assign(_target=labels)
            for label in (0, 1):
                part = chunk.loc[chunk._target.eq(label)]
                if len(part):
                    chunks.append(part.sample(n=min(len(part), 250), random_state=SEED))
    if not chunks:
        raise ValueError(f"{name}: no CSV rows found")
    sampled = pd.concat(chunks, ignore_index=True)
    sampled = pd.concat([
        group.sample(n=min(len(group), max_per_class), random_state=SEED)
        for _, group in sampled.groupby("_target")
    ], ignore_index=True)
    if set(sampled._target) != {0, 1}:
        raise ValueError(f"{name}: both normal and attack labels required; add more source files")
    return sampled

def _numeric_flow_features(train, val, test):
    excluded = {"_target", "label", "attack", "attack_cat", "category", "subcategory",
                "type", "id", "flow id", "timestamp", "ts", "stime", "ltime",
                "src ip", "dst ip", "srcip", "dstip", "saddr", "daddr", "sport", "dport"}
    fields = []
    for col in train.columns.intersection(val.columns).intersection(test.columns):
        if col.strip().lower() in excluded:
            continue
        converted = pd.to_numeric(train[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if converted.notna().mean() >= 0.7 and converted.nunique(dropna=True) > 1:
            fields.append(col)
    if not fields:
        raise ValueError("No usable shared numeric flow features")
    def convert(frame):
        return frame[fields].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return convert(train), convert(val), convert(test), fields

def run_secondary_baseline(name, paths):
    if not paths:
        return {"dataset": name, "status": "not_run", "reason": "Set local CSV path(s)"}
    if name == "UNSW-NB15":
        if len(paths) != 2:
            raise ValueError("UNSW-NB15 requires [official train CSV, official test CSV]")
        train_all = _read_csv_samples(paths[:1], name)
        test = _read_csv_samples(paths[1:], name)
        train, val = train_test_split(train_all, test_size=0.2, stratify=train_all._target,
                                      random_state=SEED)
        split_method = "official test; stratified train/validation"
    else:
        data = _read_csv_samples(paths, name)
        train_val, test = train_test_split(data, test_size=0.2, stratify=data._target,
                                          random_state=SEED)
        train, val = train_test_split(train_val, test_size=0.25, stratify=train_val._target,
                                      random_state=SEED)
        split_method = "exploratory stratified flow split"
    xtr, xv, xt, features = _numeric_flow_features(train, val, test)
    model = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(max_iter=100, random_state=SEED))
    ])
    model.fit(xtr, train._target)
    val_scores = model.predict_proba(xv)[:, 1]
    test_scores = model.predict_proba(xt)[:, 1]
    # Validate the full path while keeping test-only metrics distinct.
    if not np.isfinite(val_scores).all() or not np.isfinite(test_scores).all():
        raise ValueError(f"{name}: nonfinite prediction")
    return {
        "dataset": name, "status": "measured", "split": split_method,
        "n_train": len(train), "n_val": len(val), "n_test": len(test),
        "n_features": len(features),
        "test_roc_auc": roc_auc_score(test._target, test_scores),
        "test_pr_auc": average_precision_score(test._target, test_scores),
        "note": "binary normal/attack flow task; not web-bot class accuracy",
    }

secondary_results = pd.DataFrame([
    run_secondary_baseline(name, paths) for name, paths in SECONDARY_PATHS.items()
])
display(secondary_results)


## 12. Network coordination Graph Transformer

The default run uses the synthetic campaign export from initial ETL. Real CSV experiments can use **network flow CSVs that contain source IP, destination IP, timestamps, and a binary attack label**. CSE-CIC-IDS2018, BoT-IoT, and TON_IoT may provide suitable exports. The official UNSW-NB15 train/test feature subsets often omit identities and timestamps; use its raw flow CSVs if you want this graph experiment.

Nodes are `(source IP, five-minute window)`. An edge requires a shared destination or subnet within 15 minutes; edge attributes expose the relation and time overlap. The model reuses the edge-aware Graph Transformer above with a two-class output. Preprocessing and graphs are isolated by chronological split. The experiment fails clearly if a split lacks both classes; it never silently substitutes a random split.

**Interpretation:** the labels support normal-versus-attack flow detection. Synthetic campaign IDs support a controlled cluster-recovery test. On real datasets without campaign truth, multi-source components remain suspected coordination. Edge inference latency here measures a full graph snapshot on the notebook hardware, not per-request CDN latency. No mitigation is sent.


In [ ]:
import ipaddress

COORD_GRAPH_DATASET = "synthetic_campaign"  # or "CSE-CIC-IDS2018" / "BoT-IoT" / "TON_IoT"
COORD_GRAPH_PATHS = ([str(SYNTHETIC_CAMPAIGN_PATH)] if COORD_GRAPH_DATASET == "synthetic_campaign"
                     else SECONDARY_PATHS[COORD_GRAPH_DATASET])
COORD_WINDOW_MINUTES = 5
COORD_MAX_ROWS_PER_CLASS = 20_000

def _column(frame, options):
    normalized = {str(c).strip().lower().replace(" ", "").replace("_", ""): c
                  for c in frame.columns}
    return next((normalized[x.replace(" ", "").replace("_", "").lower()]
                 for x in options
                 if x.replace(" ", "").replace("_", "").lower() in normalized), None)

def _subnet(value):
    try:
        return str(ipaddress.ip_network(str(value) + "/24", strict=False))
    except ValueError:
        return None

def prepare_coordination_nodes(paths, dataset_name):
    if dataset_name == "synthetic_campaign":
        raw = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
        raw["_target"], _ = _binary_labels(raw, dataset_name)
    else:
        raw = _read_csv_samples(paths, dataset_name, max_per_class=COORD_MAX_ROWS_PER_CLASS)
    src_col = _column(raw, ["Src IP", "Source IP", "src_ip", "srcip", "saddr"])
    dst_col = _column(raw, ["Dst IP", "Destination IP", "dst_ip", "dstip", "daddr"])
    time_col = _column(raw, ["Timestamp", "ts", "stime", "time"])
    port_col = _column(raw, ["Dst Port", "Destination Port", "dport", "dst_port"])
    if not src_col or not dst_col or not time_col:
        raise ValueError(f"{dataset_name}: source IP, destination IP, and timestamp are required")
    if pd.api.types.is_numeric_dtype(raw[time_col]):
        stamp = pd.to_datetime(pd.to_numeric(raw[time_col], errors="coerce"),
                               unit="s", errors="coerce", utc=True)
    else:
        stamp = pd.to_datetime(raw[time_col], errors="coerce", utc=True,
                               dayfirst=(dataset_name == "CSE-CIC-IDS2018"))
    raw = raw.assign(_time=stamp, _src=raw[src_col].astype(str), _dst=raw[dst_col].astype(str))
    raw = raw.loc[raw._time.notna() & raw._src.ne("-") & raw._dst.ne("-")].copy()
    if raw.empty:
        raise ValueError(f"{dataset_name}: no usable timestamped flows")
    raw["_window"] = raw._time.dt.floor(f"{COORD_WINDOW_MINUTES}min")
    raw["_subnet"] = raw._src.map(_subnet)
    raw["_port"] = raw[port_col].astype(str) if port_col else "unknown"
    peer_counts = raw.groupby(["_window", "_dst"])._src.nunique().rename("_target_peers")
    raw = raw.join(peer_counts, on=["_window", "_dst"])
    packet_col = _column(raw, ["pkts", "Tot Fwd Pkts", "total_fwd_packets",
                               "spkts", "orig_pkts", "packets"])
    raw["_packets"] = (pd.to_numeric(raw[packet_col], errors="coerce")
                       if packet_col else 0.0)
    summary = raw.groupby(["_src", "_window"], sort=True).agg(
        flow_count=("_target", "size"), label=("_target", "max"),
        destination_count=("_dst", "nunique"), port_count=("_port", "nunique"),
        max_shared_target_sources=("_target_peers", "max"),
        mean_packets=("_packets", "mean"),
        destination_set=("_dst", lambda s: frozenset(s)),
        port_set=("_port", lambda s: frozenset(s)),
        subnet=("_subnet", "first"),
    ).reset_index().sort_values("_window").reset_index(drop=True)
    if dataset_name == "synthetic_campaign":
        truth = raw.groupby(["_src", "_window"]).campaign_id.agg(
            lambda s: s.loc[s.ne("none")].mode().iloc[0] if s.ne("none").any() else "none")
        summary = summary.join(truth.rename("campaign_id"), on=["_src", "_window"])
    summary["window_hour"] = summary._window.dt.hour.astype(float)
    return summary

COORD_FEATURES = ["flow_count", "destination_count", "port_count",
                  "max_shared_target_sources", "mean_packets", "window_hour"]
COORD_EDGE_FEATURES = ["shared_destination", "same_subnet", "port_overlap", "time_similarity"]

def split_coordination_nodes(nodes, dataset_name=None):
    times = (np.sort(nodes._window.dt.floor("D").unique()) if dataset_name == "synthetic_campaign"
             else np.sort(nodes._window.unique()))
    if len(times) < (3 if dataset_name == "synthetic_campaign" else 15):
        raise ValueError("At least 15 distinct windows required for chronological coordination evaluation")
    a, b = int(len(times)*.6), int(len(times)*.8)
    cuts = ((pd.Timestamp(times[a-1]) + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1),
             pd.Timestamp(times[b-1]) + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1))
            if dataset_name == "synthetic_campaign" else (times[a-1], times[b-1]))
    parts = {
        "train": nodes[nodes._window <= cuts[0]].reset_index(drop=True),
        "val": nodes[(nodes._window > cuts[0]) & (nodes._window <= cuts[1])].reset_index(drop=True),
        "test": nodes[nodes._window > cuts[1]].reset_index(drop=True),
    }
    if any(set(p.label) != {0, 1} for p in parts.values()):
        raise ValueError("Chronological split has a one-class period; this dataset cannot support this test")
    assert max(parts["train"]._window) < min(parts["val"]._window)
    assert max(parts["val"]._window) < min(parts["test"]._window)
    if dataset_name == "synthetic_campaign":
        campaigns = [set(p.loc[p.label.eq(1), "campaign_id"]) for p in parts.values()]
        assert all(not campaigns[i] & campaigns[j] for i,j in ((0,1),(0,2),(1,2)))
    return parts

def build_coordination_graph(frame, features):
    if not PYG_AVAILABLE:
        raise RuntimeError("Install torch-geometric for the coordination graph")
    local = frame.reset_index(drop=True)
    pairs = set()
    for key in ["subnet"]:
        for value, ids in local.groupby(key).groups.items():
            if value is None or str(value).lower() == "nan":
                continue
            members = sorted(ids, key=lambda i: local.at[i, "_window"])
            for a, b in zip(members, members[1:]):
                if (local.at[b, "_window"] - local.at[a, "_window"]).total_seconds() <= 900:
                    pairs.add((min(a, b), max(a, b)))
    # Destination fan-out is bounded by adjacent windows rather than a dense clique.
    destination_members = {}
    for i, destinations in enumerate(local.destination_set):
        for dest in destinations:
            destination_members.setdefault(dest, []).append(i)
    for members in destination_members.values():
        members.sort(key=lambda i: local.at[i, "_window"])
        for a, b in zip(members, members[1:]):
            if a != b and (local.at[b, "_window"] - local.at[a, "_window"]).total_seconds() <= 900:
                pairs.add((min(a, b), max(a, b)))
    src, dst, attrs = [], [], []
    for a, b in sorted(pairs):
        left, right = local.iloc[a], local.iloc[b]
        seconds = abs((left._window - right._window).total_seconds())
        attr = [
            float(bool(left.destination_set & right.destination_set)),
            float(pd.notna(left.subnet) and left.subnet == right.subnet),
            len(left.port_set & right.port_set) / max(1, len(left.port_set | right.port_set)),
            math.exp(-seconds / 900),
        ]
        src.extend([a, b]); dst.extend([b, a]); attrs.extend([attr, attr])
    isolated = set(range(len(local))) - set(src) - set(dst)
    for i in isolated:
        src.append(i); dst.append(i); attrs.append([0., 0., 0., 1.])
    return Data(
        x=torch.tensor(features, dtype=torch.float),
        edge_index=torch.tensor([src, dst], dtype=torch.long),
        edge_attr=torch.tensor(attrs, dtype=torch.float),
        y=torch.tensor(local.label.to_numpy(), dtype=torch.long),
    )


In [ ]:
def run_coordination_graph(paths, dataset_name, graph_epochs=25):
    nodes = prepare_coordination_nodes(paths, dataset_name)
    splits = split_coordination_nodes(nodes, dataset_name)
    scaler = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())])
    arrays = {"train": scaler.fit_transform(splits["train"][COORD_FEATURES])}
    for split in ("val", "test"):
        arrays[split] = scaler.transform(splits[split][COORD_FEATURES])
    graphs_local = {k: build_coordination_graph(splits[k], arrays[k]).to(DEVICE)
                    for k in ("train", "val", "test")}
    for k, graph in graphs_local.items():
        assert graph.num_nodes == len(splits[k])
        assert np.array_equal(graph.y.cpu().numpy(), splits[k].label.to_numpy())
        source, target = graph.edge_index.cpu().numpy()
        if len(source):
            delta = (splits[k]._window.iloc[source].to_numpy() -
                     splits[k]._window.iloc[target].to_numpy())
            assert np.all(np.abs(delta.astype("timedelta64[s]").astype(int)) <= 900)

    train_g, val_g, test_g = (graphs_local[k] for k in ("train", "val", "test"))
    model = EdgeAwareGraphTransformer(
        in_dim=len(COORD_FEATURES), edge_dim=len(COORD_EDGE_FEATURES),
        num_classes=2, hidden=64, layers=2, heads=4
    ).to(DEVICE)
    count = np.bincount(train_g.y.cpu().numpy(), minlength=2)
    weights = torch.tensor(len(train_g.y)/(2*np.maximum(count, 1)),
                           dtype=torch.float, device=DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=5e-4)
    best, best_loss = None, float("inf")
    for _ in range(graph_epochs):
        model.train(); optimizer.zero_grad()
        loss = F.cross_entropy(model(train_g), train_g.y, weight=weights)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(val_g), val_g.y).item()
        if val_loss < best_loss:
            best_loss, best = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best)
    model.eval()
    with torch.no_grad():
        val_score = model(val_g).softmax(-1)[:, 1].cpu().numpy()
        test_score = model(test_g).softmax(-1)[:, 1].cpu().numpy()
    baseline = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("model", HistGradientBoostingClassifier(max_iter=100, random_state=SEED))])
    baseline.fit(splits["train"][COORD_FEATURES], splits["train"].label)
    base_score = baseline.predict_proba(splits["test"][COORD_FEATURES])[:, 1]

    negative = np.sort(val_score[splits["val"].label.eq(0).to_numpy()])
    allowed_fp = int(np.floor(0.01 * len(negative)))
    threshold = (np.nextafter(negative[-allowed_fp-1], np.inf)
                 if allowed_fp < len(negative) else 0.0)
    alerts = test_score >= threshold
    false_positive_rate = float(alerts[splits["test"].label.eq(0).to_numpy()].mean())
    attack_recall = float(alerts[splits["test"].label.eq(1).to_numpy()].mean())

    # Components among alerted nodes; count only groups with multiple source identities.
    parent = list(range(len(splits["test"])))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    src, dst = test_g.edge_index.cpu().numpy()
    for a, b in zip(src, dst):
        if alerts[a] and alerts[b]:
            parent[root(a)] = root(b)
    groups = {}
    for i in np.where(alerts)[0]:
        groups.setdefault(root(int(i)), []).append(int(i))
    multi_source = [ids for ids in groups.values() if splits["test"].iloc[ids]._src.nunique() > 1]
    multi_source_components = len(multi_source)
    campaign_metrics = {}
    if "campaign_id" in splits["test"]:
        true_campaigns = set(splits["test"].loc[splits["test"].label.eq(1), "campaign_id"])
        recovered = set()
        pure_components = 0
        for ids in multi_source:
            members = splits["test"].iloc[ids]
            counts = members.loc[members.campaign_id.ne("none"), "campaign_id"].value_counts()
            if len(counts) and counts.iloc[0] >= 2 and counts.iloc[0] / len(members) >= .8:
                pure_components += 1
                recovered.add(counts.index[0])
        campaign_metrics = {"held_out_campaigns":len(true_campaigns),
                            "recovered_campaigns":len(recovered & true_campaigns),
                            "campaign_recall":len(recovered & true_campaigns)/len(true_campaigns),
                            "component_precision_at_80pct_purity":pure_components/max(1,multi_source_components)}

    return {
        "dataset": dataset_name, "status": "measured",
        "train_nodes": len(train_g.y), "val_nodes": len(val_g.y), "test_nodes": len(test_g.y),
        "test_edges": test_g.num_edges,
        "graph_roc_auc": roc_auc_score(splits["test"].label, test_score),
        "graph_pr_auc": average_precision_score(splits["test"].label, test_score),
        "tabular_roc_auc": roc_auc_score(splits["test"].label, base_score),
        "tabular_pr_auc": average_precision_score(splits["test"].label, base_score),
        "validation_threshold": float(threshold),
        "test_fpr": false_positive_rate, "test_attack_recall": attack_recall,
        "alerted_multi_source_components": multi_source_components,
        **campaign_metrics,
        "snapshot_p95_ms": latency_percentiles(lambda: model(test_g), repeats=30)["p95_ms"],
    }

if COORD_GRAPH_PATHS:
    coordination_result = run_coordination_graph(COORD_GRAPH_PATHS, COORD_GRAPH_DATASET)
else:
    coordination_result = {
        "dataset": COORD_GRAPH_DATASET, "status": "not_run",
        "reason": "Set COORD_GRAPH_PATHS to local flow CSVs with source IP, destination IP, timestamp, and label"
    }
display(pd.DataFrame([coordination_result]))


### Serving interpretation

The tabular model is the candidate low-latency scorer. The coordination graph is an offline or asynchronous signal until a serving implementation can build neighborhoods within a measured edge budget. A missing graph score should return the tabular/allow decision according to an explicit policy; the notebook does not implement a CDN policy service or claim a reduction in incident MTTM. Evaluate graph lift, false positives, drift, and latency on each real source before connecting it to mitigation.


## 13. Real-data limitations

- This notebook loads the official phase 1 web logs and published session labels at run time.
- The original authors' session annotations are joined by PHP session ID; sessions never cross train, validation, or test.
- Humans, moderate bots, and advanced bots were collected on different dates. Their classes are correlated with collection conditions.
- The logs do not label legitimate crawlers, benign automation, or abusive intent. They do not expose network coordination signals suitable for Sybil detection.
- For a true four-class Traffic Integrity study, collect authorized, contemporaneous requests from humans, verified crawlers, known benign API clients, and confirmed abusive automation, with reviewable identity and outcome labels. Then use a chronological holdout and measured operational thresholds.

Do not present this benchmark as proof of a production four-class traffic classifier. It is a reproducible real-data web-bot modeling foundation.

- Synthetic campaign metrics validate the code path only, not performance on real Sybil actors.

The four-class exercise below is a **synthetic integration test**. Independent authorization, crawler verification, analyst adjudication, and outcome measurement require a separately collected real dataset.


## 14. Four-class traffic study: controlled, provenance-aware experiment

The Bournemouth annotations provide three behavioral collection classes and cannot certify authorized crawlers, benign API clients, or abusive intent. This independent synthetic fixture exercises the complete four-class ETL and evaluation contract. Each row has a claimed identity, a reviewable outcome code, and provenance `synthetic_fixture`. These are scenario assumptions, **not independently verified identities or real adjudications**. A production run must replace the fixture with authorized, contemporaneous logs and reviewed labels.

The fixture generates all classes on each day, with overlapping request rates, shared user agents, destination paths, and false positives in risk signals. Client IDs and outcome fields are withheld from features. Days 1–6 train, 7–8 validate, and 9–10 test. Thresholds are selected exclusively on validation data.


In [ ]:
FOUR_CLASS_NAMES = {0: "human", 1: "verified_crawler", 2: "benign_api", 3: "abusive_automation"}
FOUR_CLASS_PATH = Path("data/synthetic_four_class_requests.csv")

def generate_four_class_requests(path=FOUR_CLASS_PATH, seed=829):
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2026-03-01T00:00:00Z")
    # Class profiles intentionally overlap. Claim and adjudication are evaluation-only fields.
    profiles = {
        0: (2.8, 1.1, .035), 1: (4.1, 1.5, .018),
        2: (4.5, 1.6, .025), 3: (5.2, 2.0, .13),
    }
    for day in range(10):
        for label in FOUR_CLASS_NAMES:
            for client in range(30):
                identity = f"day{day:02d}-class{label}-actor{client:03d}"
                count_mean, count_sd, error_p = profiles[label]
                for window in rng.choice(72, size=int(rng.integers(8, 16)), replace=False):
                    count = max(1, int(rng.normal(count_mean, count_sd)))
                    window_start = start + pd.Timedelta(days=day, hours=9, minutes=5*int(window))
                    for j in range(count):
                        ts = window_start + pd.Timedelta(seconds=int(rng.integers(0,300)))
                        rows.append({
                            "timestamp":ts, "client_id":identity, "session_id":f"{identity}-w{window:02d}",
                            "path":str(rng.choice(["/", "/search", "/product", "/api/items", "/api/profile"],
                                                   p=([.24,.31,.25,.16,.04] if label==0 else
                                                      [.08,.18,.30,.35,.09] if label==1 else
                                                      [.04,.08,.20,.51,.17] if label==2 else
                                                      [.08,.19,.27,.38,.08]))),
                            "user_agent":str(rng.choice(["Chrome", "PythonHTTP", "MobileSDK", "Crawler"],
                                                   p=([.83,.02,.14,.01] if label==0 else
                                                      [.08,.14,.08,.70] if label==1 else
                                                      [.07,.62,.29,.02] if label==2 else
                                                      [.45,.35,.15,.05]))),
                            "status_code":int(429 if rng.random()<error_p else 200),
                            "response_bytes":int(rng.lognormal(7.2, .65)),
                            "label":label, "claimed_identity":f"claim-{identity}",
                            "outcome_code":("synthetic_abuse_confirmed" if label==3 else
                                            "synthetic_authorized"),
                            "label_provenance":"synthetic_fixture",
                        })
    frame=pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    path.parent.mkdir(parents=True,exist_ok=True)
    frame.to_csv(path,index=False)
    return frame

_generated_four_requests=generate_four_class_requests()
# Exercise the exported CSV ETL, including timestamp parsing and schema checks.
four_requests=pd.read_csv(FOUR_CLASS_PATH)
four_requests["timestamp"]=pd.to_datetime(four_requests.timestamp,errors="coerce",utc=True)
required_four={"timestamp","client_id","session_id","path","user_agent","status_code",
               "response_bytes","label","claimed_identity","outcome_code","label_provenance"}
assert required_four.issubset(four_requests.columns)
assert four_requests.timestamp.notna().all()
assert four_requests.client_id.notna().all() and four_requests.session_id.notna().all()
assert four_requests.label.notna().all() and four_requests.label.between(0,3).all()
assert len(four_requests)==len(_generated_four_requests)
del _generated_four_requests
assert set(four_requests.label)==set(FOUR_CLASS_NAMES)
assert four_requests.groupby(four_requests.timestamp.dt.floor("D")).label.nunique().eq(4).all()
assert four_requests.groupby("client_id").label.nunique().eq(1).all()
assert four_requests.label_provenance.eq("synthetic_fixture").all()
print("Four-class synthetic ETL:",len(four_requests),"requests,",four_requests.client_id.nunique(),"clients")


### Chronological holdout, modeling, and operational checks

The model sees per-client five-minute behavior only. IPs, client identifiers, claimed identities, outcomes, and provenance are retained for audit but excluded from its features. The abusive score is used for a validation-selected threshold under a 2% false positive rate budget. Counts and rates on the later test period are reported without retuning.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

four_requests["window_start"]=four_requests.timestamp.dt.floor("5min")
four_nodes=(four_requests.groupby(["client_id","window_start"],as_index=False)
 .agg(label=("label","first"), request_count=("path","size"),
      unique_paths=("path","nunique"), api_fraction=("path",lambda s:s.str.startswith("/api").mean()),
      error_rate=("status_code",lambda s:s.ge(400).mean()),
      mean_bytes=("response_bytes","mean"),
      automation_ua_fraction=("user_agent",lambda s:s.isin(["PythonHTTP","Crawler"]).mean())))
FOUR_FEATURES=["request_count","unique_paths","api_fraction","error_rate", "mean_bytes","automation_ua_fraction"]
days=np.sort(four_nodes.window_start.dt.floor("D").unique())
assert len(days)==10
four_splits={
 "train":four_nodes[four_nodes.window_start.dt.floor("D").isin(days[:6])].copy(),
 "val":four_nodes[four_nodes.window_start.dt.floor("D").isin(days[6:8])].copy(),
 "test":four_nodes[four_nodes.window_start.dt.floor("D").isin(days[8:])].copy(),
}
assert max(four_splits["train"].window_start)<min(four_splits["val"].window_start)
assert max(four_splits["val"].window_start)<min(four_splits["test"].window_start)
assert all(set(four_splits[a].client_id).isdisjoint(four_splits[b].client_id)
           for a,b in [("train","val"),("train","test"),("val","test")])
assert all(set(s.label)==set(FOUR_CLASS_NAMES) for s in four_splits.values())
assert not set(FOUR_FEATURES)&{"client_id","claimed_identity","outcome_code","label_provenance","window_start","label"}
four_model=Pipeline([("impute",SimpleImputer(strategy="median")),
                     ("classifier",HistGradientBoostingClassifier(max_iter=100,random_state=SEED))])
four_model.fit(four_splits["train"][FOUR_FEATURES],four_splits["train"].label)
val_prob=four_model.predict_proba(four_splits["val"][FOUR_FEATURES])
test_prob=four_model.predict_proba(four_splits["test"][FOUR_FEATURES])
assert list(four_model.classes_)==list(FOUR_CLASS_NAMES)
val_nonabusive=val_prob[four_splits["val"].label.ne(3).to_numpy(),3]
# Strictly above the validation benign quantile; report actual rate because of ties.
operating_threshold=float(np.nextafter(np.quantile(val_nonabusive,.98,method="higher"),np.inf))
test_alert=test_prob[:,3]>=operating_threshold
test_labels=four_splits["test"].label.to_numpy()
val_fpr=float((val_nonabusive>=operating_threshold).mean())
test_fpr=float(test_alert[test_labels!=3].mean())
test_recall=float(test_alert[test_labels==3].mean())
print("Split nodes:",{k:len(v) for k,v in four_splits.items()})
print("Synthetic four-class test classification:")
print(classification_report(test_labels,test_prob.argmax(axis=1),target_names=list(FOUR_CLASS_NAMES.values()),digits=3))
print("Confusion matrix (rows=true, columns=predicted):\n",confusion_matrix(test_labels,test_prob.argmax(axis=1)))
print("Abusive threshold:",operating_threshold,"validation FPR:",val_fpr,
      "test FPR:",test_fpr,"test recall:",test_recall)
assert 0<=val_fpr<=.02 and np.isfinite(test_prob).all()
print("Test false positives:",int(test_alert[test_labels!=3].sum()),"/",int((test_labels!=3).sum()),
      "true positives:",int(test_alert[test_labels==3].sum()),"/",int((test_labels==3).sum()))


## 15. Reviewed four-class CSV contract and execution gate

A real four-class run requires authorized, contemporaneous traffic with independently reviewed identity and outcome evidence. Set `REVIEWED_FOUR_CLASS_CSV` to a local CSV only after collection. The strict loader rejects synthetic provenance, unreviewed rows, missing source evidence, conflicting labels per client, and missing classes in any chronological period. It never silently relabels Bournemouth sessions or substitutes the synthetic fixture. Its training features intentionally exclude identity, review, and outcome fields. Campaign-ground-truth and operational deployment remain separate requirements.

Required columns: `timestamp`, `client_id`, `session_id`, `path`, `user_agent`, `status_code`, `response_bytes`, `label` (0–3), `claimed_identity`, `outcome_code`, `label_provenance`, `review_status`, `source_reference`. Review status must be `approved`; provenance must be `reviewed_real`. `source_reference` points to the record used for the label; this code validates presence, not the underlying human review.


In [ ]:
REVIEWED_FOUR_CLASS_CSV = None  # e.g. "data/reviewed_authorized_traffic.csv"

def load_reviewed_four_class_csv(path):
    frame=pd.read_csv(path,low_memory=False)
    required={"timestamp","client_id","session_id","path","user_agent", "status_code",
              "response_bytes","label","claimed_identity","outcome_code",
              "label_provenance","review_status","source_reference"}
    missing=required-set(frame.columns)
    if missing: raise ValueError(f"Reviewed CSV missing: {sorted(missing)}")
    if frame.empty: raise ValueError("Reviewed CSV is empty")
    frame["timestamp"]=pd.to_datetime(frame.timestamp,errors="coerce",utc=True)
    frame["label"]=pd.to_numeric(frame.label,errors="raise")
    frame["status_code"]=pd.to_numeric(frame.status_code,errors="raise")
    frame["response_bytes"]=pd.to_numeric(frame.response_bytes,errors="raise")
    if frame.timestamp.isna().any() or frame.label.isna().any() or not frame.label.isin(FOUR_CLASS_NAMES).all():
        raise ValueError("Invalid timestamp or four-class label")
    if not frame.review_status.eq("approved").all() or not frame.label_provenance.eq("reviewed_real").all():
        raise ValueError("All rows require approved review and reviewed_real provenance")
    if frame[list(required-{"timestamp","label","status_code","response_bytes"})].isna().any().any():
        raise ValueError("Missing identity, outcome, evidence, or request fields")
    if frame.source_reference.astype(str).str.strip().eq("").any():
        raise ValueError("Empty source evidence reference")
    if frame.groupby("client_id").label.nunique().gt(1).any():
        raise ValueError("Conflicting client labels require adjudication")
    if frame.groupby("session_id").client_id.nunique().gt(1).any():
        raise ValueError("Session reused across clients")
    frame["label"]=frame.label.astype(int)
    return frame.sort_values("timestamp").reset_index(drop=True)

def evaluate_reviewed_four_class(frame):
    data=frame.copy()
    data["window_start"]=data.timestamp.dt.floor("5min")
    nodes=(data.groupby(["client_id","window_start"],as_index=False)
     .agg(label=("label","first"), request_count=("path","size"),
          unique_paths=("path","nunique"), api_fraction=("path",lambda s:s.str.startswith("/api").mean()),
          error_rate=("status_code",lambda s:s.ge(400).mean()), mean_bytes=("response_bytes","mean"),
          automation_ua_fraction=("user_agent",lambda s:s.isin(["PythonHTTP","Crawler"]).mean())))
    days=np.sort(nodes.window_start.dt.floor("D").unique())
    if len(days)<5: raise ValueError("At least five distinct collection days are required")
    a,b=int(len(days)*.6),int(len(days)*.8)
    if min(a,b-a,len(days)-b)<1: raise ValueError("Chronological split is empty")
    parts={"train":nodes[nodes.window_start.dt.floor("D").isin(days[:a])],
           "val":nodes[nodes.window_start.dt.floor("D").isin(days[a:b])],
           "test":nodes[nodes.window_start.dt.floor("D").isin(days[b:])]}
    if any(set(part.label)!=set(FOUR_CLASS_NAMES) for part in parts.values()):
        raise ValueError("Every chronological split requires all four reviewed classes")
    if any(not set(parts[x].client_id).isdisjoint(parts[y].client_id)
           for x,y in (("train","val"),("train","test"),("val","test"))):
        raise ValueError("Client crosses chronological split; use a client-disjoint collection or cohort")
    model=Pipeline([("impute",SimpleImputer(strategy="median")),
                    ("classifier",HistGradientBoostingClassifier(max_iter=100,random_state=SEED))])
    model.fit(parts["train"][FOUR_FEATURES],parts["train"].label)
    val=model.predict_proba(parts["val"][FOUR_FEATURES])
    test=model.predict_proba(parts["test"][FOUR_FEATURES])
    valid=val[parts["val"].label.ne(3).to_numpy(),3]
    threshold=float(np.nextafter(np.quantile(valid,.98,method="higher"),np.inf))
    target=parts["test"].label.to_numpy(); alert=test[:,3]>=threshold
    return {"status":"measured_reviewed_real", "train_nodes":len(parts["train"]),
            "val_nodes":len(parts["val"]),"test_nodes":len(parts["test"]),
            "test_accuracy":float((test.argmax(1)==target).mean()),
            "validation_fpr":float((valid>=threshold).mean()),
            "test_fpr":float(alert[target!=3].mean()),
            "test_abusive_recall":float(alert[target==3].mean()),
            "threshold":threshold}

if REVIEWED_FOUR_CLASS_CSV:
    reviewed_result=evaluate_reviewed_four_class(load_reviewed_four_class_csv(REVIEWED_FOUR_CLASS_CSV))
else:
    reviewed_result={"status":"not_run", "reason":"No independently reviewed, authorized four-class CSV supplied"}
display(pd.DataFrame([reviewed_result]))
